# Rolling-year results analysis

This notebook uses the repository-wide analysis schema. In addition to the common results, it reports checkpoint handoffs, monthly modeled-versus-observed generation, and resource seasonality.

In [ ]:
# ===================== USER SETTINGS =====================
RESULT_PATH = 'results/run_results/rolling_2025-04-01_2026-03-31_relaxed_uc'
OUTPUT_DIR = None  # None writes below RESULT_PATH/analysis
SAVE_TABLES_AND_FIGURES = True
# =========================================================

## Load retained rolling checkpoints

In [ ]:
from pathlib import Path
import sys

from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent
if not (ROOT / 'scripts').exists():
    raise FileNotFoundError('Run this notebook from the repository root or notebooks folder.')
sys.path.insert(0, str(ROOT / 'scripts'))

from results_analysis import analyze_run, export_analysis, load_analysis_run, load_observed_generation

run = load_analysis_run(ROOT / RESULT_PATH)
observed = load_observed_generation()
analysis = analyze_run(run, observed)
print(f'Loaded {len(run.networks)} checkpoints: {run.snapshots[0]} to {run.snapshots[-1]} ({len(run.snapshots):,} hours)')
print(f"Unit commitment: {run.metadata.get('unit_commitment', 'unknown')}")

## Validation and boundary handoffs

In [ ]:
display(analysis.tables['chronology_validation'])
display(analysis.tables['topology_validation'])
display(analysis.tables['boundary_handoffs'].round(6))
display(analysis.tables['system_metrics'].round(3))

## Generation, storage, commitment, and transmission

In [ ]:
display(analysis.tables['generation_comparison'].round(2))
display(analysis.tables['storage_metrics'].round(2))
display(analysis.tables['store_metrics'].round(2))
display(analysis.tables['commitment_by_carrier'].round(2))
display(analysis.tables['corridor_metrics'].head(12).round(2))

## Monthly comparison and resource seasonality

In [ ]:
display(analysis.tables['monthly_generation'].round(3))
display(analysis.tables['monthly_resource_seasonality'].round(4))
display(analysis.tables['seasonality_summary'].round(3))
display(analysis.tables['seasonality_fit'].round(3))

## Standard figures

In [ ]:
for name, figure in analysis.figures.items():
    print(name.replace('_', ' ').title())
    display(figure)

## Export standardized artifacts

In [ ]:
if SAVE_TABLES_AND_FIGURES:
    destination = export_analysis(
        run, analysis, None if OUTPUT_DIR is None else ROOT / OUTPUT_DIR
    )
    print(f'Analysis written to {destination}')